In [67]:
import pertpy as pt
from bicycle.model import BICYCLE
import numpy as np

import time
import os
from pathlib import Path
from os import environ
import pytorch_lightning as pl
import torch
from bicycle.dictlogger import DictLogger
from bicycle.model import BICYCLE
from bicycle.utils.data import (
    get_diagonal_mask,
    compute_inits,
)
from bicycle.utils.plotting import plot_training_results
from pytorch_lightning.callbacks import RichProgressBar, StochasticWeightAveraging
from bicycle.callbacks import CustomModelCheckpoint, GenerateCallback, MyLoggerCallback
import numpy as np
import pandas as pd

import torch

# Check if MPS is available
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS device found! Using Apple Silicon GPU.")
else:
    device = torch.device("cpu")
    print("MPS not available. Using CPU.")
device = torch.device("cpu")
# Move a tensor to the MPS device
x = torch.ones(3, 3, device=device)
print(x)


MPS device found! Using Apple Silicon GPU.
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])


In [25]:
from typing import Optional, Sequence, Union
import numpy as np
import pandas as pd
import torch
from anndata import AnnData
from bicycle.utils.data import (
    compute_inits,
    create_data,
    create_loaders,
    get_diagonal_mask,
)

CONTROL_LABELS = {"control", "ctrl", "non-targeting"}
import pandas as pd
def pert_to_gene(
    label,
    control_substrings=("(mod)",),
    guide_prefixes=("pDS", "pBA"),
):
    """Map 'OST4_pDS353' -> 'OST4', controls -> 'control', missing/* -> None."""
    if pd.isna(label):
        return None
    s = str(label).strip()
    if s in ("*", "nan", "None", "", "<NA>") or s.startswith("*"):
        return None
    if s.lower() in CONTROL_LABELS or any(sub in s for sub in control_substrings):
        return "control"
    if "_" not in s:
        return s
    gene, guide = s.rsplit("_", 1)
    if not guide.startswith(tuple(guide_prefixes)):
        return None
    return gene
def prepare_bicycle_from_adata(
    adata,
    perturbation_key="perturbation",
    extra_genes=None,
    drop_unassigned=True,
    collapse_guides=True,
    control_first=True,
):
    ad = adata.copy()
    pert = ad.obs[perturbation_key]
    if drop_unassigned:
        keep = pert.notna() & (pert.astype(str) != "*")
        ad = ad[keep].copy()
        pert = ad.obs[perturbation_key]
    target = pert.map(pert_to_gene)
    unparsed = pert[target.isna()]
    if len(unparsed):
        if drop_unassigned:
            ad = ad[target.notna()].copy()
            target = ad.obs[perturbation_key].map(pert_to_gene)
        else:
            raise ValueError(
                f"Could not parse perturbation labels: {unparsed.unique().tolist()}"
            )
    ad.obs["target_gene"] = target.to_numpy()
    perturbed = sorted({g for g in ad.obs["target_gene"] if g != "control"})
    gene_set = set(perturbed)
    if extra_genes is not None:
        gene_set |= set(extra_genes)
    missing_from_var = sorted(g for g in perturbed if g not in ad.var_names)
    if missing_from_var:
        raise ValueError(
            "Perturbed gene symbols not in adata.var_names. "
            "Map Ensembl IDs to symbols first. Missing: "
            f"{missing_from_var[:20]}"
        )
    ad = ad[:, ad.var_names.isin(gene_set)].copy()
    ad = ad[ad.obs["target_gene"].isin(set(ad.var_names) | {"control"})].copy()
    genes = list(ad.var_names)
    gene_to_row = {g: i for i, g in enumerate(genes)}
    perturbed_in_matrix = [g for g in genes if g in set(perturbed)]
    if collapse_guides:
        conditions = (["control"] if control_first else []) + perturbed_in_matrix
        if not control_first:
            conditions = perturbed_in_matrix + ["control"]
        cell_condition = ad.obs["target_gene"].astype(str)
    else:
        # one column per original guide label; control labels share one column
        cell_condition = [
            "control" if g == "control" else str(p)
            for g, p in zip(ad.obs["target_gene"], ad.obs[perturbation_key])
        ]
        uniq = pd.Index(cell_condition).unique().tolist()
        controls = [c for c in uniq if c == "control"]
        rest = [c for c in uniq if c != "control"]
        conditions = (controls + rest) if control_first else (rest + controls)
    cond_map = {c: i for i, c in enumerate(conditions)}
    n_genes, n_conditions = len(genes), len(conditions)
    gt_interv = torch.zeros((n_genes, n_conditions), dtype=torch.float32)
    for j, cond in enumerate(conditions):
        if cond == "control":
            continue
        gene = pert_to_gene(cond) if not collapse_guides else cond
        if gene is None or gene == "control":
            continue
        if gene not in gene_to_row:
            raise KeyError(f"Condition {cond!r} maps to {gene!r}, not in modelled genes")
        gt_interv[gene_to_row[gene], j] = 1.0
    regimes = torch.tensor([cond_map[c] for c in cell_condition], dtype=torch.long)
    X = ad.X.toarray() if hasattr(ad.X, "toarray") else np.asarray(ad.X)
    samples = torch.tensor(X, dtype=torch.float32)
    if gt_interv[:, cond_map["control"]].sum() != 0:
        raise RuntimeError("Control column of gt_interv is not all zeros")
    if collapse_guides and int(gt_interv.sum()) != len(perturbed_in_matrix):
        raise RuntimeError("Expected one 1 per perturbed gene when collapsing guides")
    return {
        "samples": samples,
        "gt_interv": gt_interv,
        "regimes": regimes,
        "genes": genes,
        "conditions": conditions,
        "adata": ad,
        "target_gene": ad.obs["target_gene"].to_numpy(),
        "cond_map": cond_map,
    }

In [13]:
adata = pt.data.adamson_2016_upr_perturb_seq()

In [26]:
adata.obs

,perturbation,read count,UMI count,tissue_type,cell_line,cancer,disease,perturbation_type,celltype,organism,ncounts,ngenes,percent_mito,percent_ribo,nperts
cell_barcode,,,,,,,,,,,,,,,
AAACATACAAGATG,63(mod)_pBA580,282.0,8.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,8866.0,2914,4.917663,21.306112,2
AAACATACACCTAG,OST4_pDS353,331.0,7.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,13785.0,3818,4.468626,19.492201,2
AAACATACTTCCCG,SEC61A1_pDS031,285.0,10.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,7569.0,2616,5.060113,23.199894,2
AAACATTGAAACAG,EIF2B4_pDS491,1036.0,30.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,13834.0,3488,5.052769,28.733555,2
AAACATTGCAGCTA,SRPR_pDS482,863.0,25.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,15507.0,3620,4.514091,26.729864,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGCATGCTTTAC,STT3A_pDS011,476.0,17.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,14524.0,3356,5.996971,22.679703,2
TTTGCATGGAGGAC,ARHGAP22_pDS458,539.0,19.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,11685.0,2961,4.612751,26.983313,2
TTTGCATGTAGAGA,63(mod)_pBA580,647.0,35.0,cell_line,K562,True,chronic myelogenous leukemia,CRISPR,lymphoblasts,human,16610.0,3473,7.242625,26.207104,2


In [27]:
labels = adata.var.index.values

In [28]:
adata.obs["perturbation"].value_counts().head(15) 

perturbation
63(mod)_pBA580        6010
Gal4-4(mod)_pBA582    1283
IER3IP1_pDS002        1222
SEC61B_pDS033         1185
ASCC3_pDS052          1142
DNAJC19_pDS026        1000
HSPA9_pDS088           894
HSPA5_pDS017           881
SAMM50_pDS156          817
YIPF5_pDS186           817
TARS_pDS405            803
XRN1_pDS411            788
GBF1_pDS043            760
TIMM23_pDS284          757
DAD1_pDS499            752
Name: count, dtype: int64

In [29]:
out = prepare_bicycle_from_adata(adata)
samples = out["samples"]       # cells x genes
gt_interv = out["gt_interv"]   # genes x conditions
regimes = out["regimes"]  

In [30]:
# DATA GENERATION
n_genes = samples.shape[1]
rank_w_cov_factor = n_genes  # Same as dictys: #min(TFs, N_GENES-1)
graph_type = "erdos-renyi"
edge_assignment = "random-uniform"
sem = "linear-ou"
graph_kwargs = {
    "abs_weight_low": 0.25,
    "abs_weight_high": 0.95,
    "p_success": 0.5,
    "expected_density": 2,
    "noise_scale": 0.5,
    "intervention_scale": 0.1,
}
n_additional_entries = 12
n_contexts = n_genes + 1  # Number of contexts
n_samples_control = 500
n_samples_per_perturbation = 250
perfect_interventions = True
make_contractive = True
make_counts = True
synthetic_T = 1.0
library_size_range = [10 * n_genes, 100 * n_genes]

In [68]:
# TRAINING

lr = 1e-3  # 3e-4
batch_size = 10_000
USE_INITS = False
use_encoder = False
n_epochs = 1000
early_stopping = False
early_stopping_patience = 500
early_stopping_min_delta = 0.01
# Maybe this helps to stop the loss from growing late during training (see current version
# of Plot_Diagnostics.ipynb)
optimizer = "adam"  # "rmsprop" #"adam"
optimizer_kwargs = {"betas": [0.5, 0.9]}  # Faster decay for estimates of gradient and gradient squared
gradient_clip_val = 1.0
GPU_DEVICE = 0
plot_epoch_callback = 1
validation_size = 0.2
lyapunov_penalty = True
swa = 250

LOGO = []
train_gene_ko = [str(x) for x in set(range(0, n_genes)) - set(LOGO)]  # We start counting at 0
# FIXME: There might be duplicates...
ho_perturbations = sorted(
    list(set([tuple(sorted(np.random.choice(n_genes, 2, replace=False))) for _ in range(0, 20)]))
)
test_gene_ko = [f"{x[0]},{x[1]}" for x in ho_perturbations]

# MODEL
x_distribution = "Multinomial"
x_distribution_kwargs = {}
model_T = 1.0
learn_T = False
use_latents = make_counts

# MINE TODO
scale_kl = 0.1   # try 0.1, then 0.01
scale_l1 = 0.1   # 1.0 is strong for ~90 genes
scale_lyapunov = 0.1
scale_spectral = 0.0
beta=1
intervention_type_inference = "dCas9"


n_factors=0
# Create Mask
mask = get_diagonal_mask(n_genes, device)

covariates = None

# if n_factors > 0:
#     mask = None

In [69]:
model = BICYCLE(
    lr,
    gt_interv,
    n_genes,
    n_samples=len(samples),
    lyapunov_penalty=lyapunov_penalty,
    perfect_interventions=True,
    rank_w_cov_factor=rank_w_cov_factor,
    init_tensors=init_tensors if USE_INITS else None,
    optimizer=optimizer,
    device=device,
    scale_l1=scale_l1,
    scale_lyapunov=scale_lyapunov,
    scale_spectral=scale_spectral,
    scale_kl=scale_kl,
    early_stopping=early_stopping,
    early_stopping_min_delta=early_stopping_min_delta,
    early_stopping_patience=early_stopping_patience,
    early_stopping_p_mode=True,
    x_distribution=x_distribution,
    mask=mask,
    use_encoder=use_encoder,
    gt_beta=None,
    use_latents=use_latents,
)
model = model.to(device)

In [70]:
import time
import pytorch_lightning as pl
import torch
from argparse import Namespace
from bicycle.dictlogger import DictLogger

def _log_hyperparams(self, params):
    if isinstance(params, dict):
        self.hyperparams = dict(params)
    elif isinstance(params, Namespace):
        self.hyperparams = vars(params)
    else:
        self.hyperparams = dict(params)
DictLogger.log_hyperparams = _log_hyperparams


class EpochMetricsCallback(pl.Callback):
    def __init__(self, stage="train"):
        self.stage = stage
        self.t0 = None
    def on_train_start(self, trainer, pl_module):
        self.t0 = time.time()
    def on_train_epoch_end(self, trainer, pl_module):
        m = trainer.callback_metrics
        def get(name):
            v = m.get(name)
            return float(v.detach().cpu()) if torch.is_tensor(v) else v
        elapsed = time.time() - self.t0
        print(
            f"[{self.stage}] epoch {trainer.current_epoch:4d} | "
            f"loss={get('train_loss')}  "
            f"nll={get('train_nll_train')}  "
            f"valid_nll={get('valid_nll_valid')}  "
            f"valid_loss={get('valid_loss')}  "
            f"kl={get('train_kl_train')}  "
            f"l1={get('train_l1')}  "
            f"lyap={get('train_lyapunov')}  "
            f"time={elapsed:.0f}s"
        )
dlogger = DictLogger()
loggers = [dlogger]

callbacks = [
    # RichProgressBar(refresh_rate=1),
    # EpochMetricsCallback(),
    GenerateCallback(
        "./out.png",
        plot_epoch_callback=plot_epoch_callback,
        true_beta=None,
        labels=out["genes"],
    ),
]
# if swa > 0:
#     callbacks.append(StochasticWeightAveraging(0.01, swa_epoch_start=swa))
    
# if CHECKPOINTING:
#     Path(os.path.join(MODEL_PATH, file_dir)).mkdir(parents=True, exist_ok=True)
#     callbacks.append(
#         CustomModelCheckpoint(
#             dirpath=os.path.join(MODEL_PATH, file_dir),
#             filename="{epoch}",
#             save_last=True,
#             save_top_k=1,
#             verbose=VERBOSE_CHECKPOINTING,
#             monitor="train_loss", ### FIXME: valid_loss
#             mode="min",
#             save_weights_only=True,
#             start_after=1000,
#             save_on_train_epoch_end=True, ### FIXME: False
#             every_n_epochs=500,
#         )
#     )
#     callbacks.append(MyLoggerCallback(dirpath=os.path.join(MODEL_PATH, file_dir)))

log_every_n_steps=10
CHECKPOINTING = False
check_val_every_n_epoch = 20
MODEL_PATH = "./model.pth"
trainer = pl.Trainer(
    max_epochs=n_epochs,
    accelerator='cpu',  # if str(device).startswith("cuda") else "cpu",
    # devices=1,
    logger=loggers,
    log_every_n_steps=log_every_n_steps,
    enable_model_summary=True,
    enable_progress_bar=True,
    enable_checkpointing=CHECKPOINTING,
    check_val_every_n_epoch=check_val_every_n_epoch,
    # devices=[GPU_DEVICE],  # if str(device).startswith("cuda") else 1,
    num_sanity_val_steps=0,
    callbacks=callbacks,
    gradient_clip_val=gradient_clip_val,
    default_root_dir=str(MODEL_PATH),
    gradient_clip_algorithm="value",
)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/mkojro/miniconda3/envs/sci/lib/python3.14/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [71]:
from bicycle.utils.data import create_loaders_norman
n_conditions = gt_interv.shape[1]
# e.g. last 10 perturbed genes as test; 0 is control
train_regimes = list(range(n_conditions - 10))
test_regimes = list(range(n_conditions - 10, n_conditions))
train_loader, validation_loader, test_loader = create_loaders_norman(
    samples,
    regimes,
    validation_size=0.2,
    batch_size=4096*2,
    SEED=0,
    train_regimes=train_regimes,
    test_regimes=test_regimes,
)
train_loader.pin_memory = False
validation_loader.pin_memory = False

# 3. Add background workers and keep them alive across epochs
# (4 is a safe starting point for Mac; do not exceed your total CPU core count)
train_loader.num_workers = 4
# train_loader.persistent_workers = True

validation_loader.num_workers = 4
# validation_loader.persistent_workers = True

if test_loader is not None:
    test_loader.pin_memory = False
    test_loader.num_workers = 4
    # test_loader.persistent_workers = True